# 8. Thời gian và tài nguyên parser

## 8.1 Thiết lập thực nghiệm

Parser được chạy mới trên snapshot cố định, output riêng và không dùng cache của lượt
trước. Phạm vi gồm xử lý từng publication, ghi ba store, checkpoint và validation cuối.
Input giữ cả version empty và version nguồn lỗi để coverage phản ánh tập M1 thực sự có.

Lượt mới được đo trên Windows CPU. Thông tin hệ điều hành, Python, thư viện và RAM được
công bố bên dưới. Đây không phải benchmark M2 cuối trên Colab CPU-only.

In [ ]:
from pathlib import Path
import json
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown
data = json.loads((Path.cwd().parent / 'report_summary.json').read_text(encoding='utf-8'))
scope, m1, m2 = data['scope'], data['m1'], data['m2']
plt.rcParams.update({'figure.figsize': (9, 4), 'axes.spines.top': False, 'axes.spines.right': False})
print(f"Snapshot: {scope['papers']} paper / {scope['versions']} version; báo cáo ngày {data['report_date']}")

In [ ]:
run=data['parser_measurement']
display(pd.Series(run['environment'],name='value').to_frame())
display(pd.DataFrame([{k:run[k] for k in ['status','runtime_sec','processed_papers','avg_sec_per_publication','final_output_bytes']}]))

## 8.2 Runtime và trung bình theo publication

Wall time được chia cho số publication thực sự xử lý mới. `parse_publication` hiện gom
gathering, hierarchy, bibliography, normalization và dedup trong một pha cấp publication;
không có số đo tách riêng cho từng thuật toán đó. Storage và validation có pha riêng.

Một run resume dùng lại output không được so sánh trực tiếp với run mới nếu không công bố
số publication mới và số đã reuse. Phân bố độ dài TeX, số version và số references cũng
ảnh hưởng runtime, nên báo cáo không ngoại suy thời gian toàn corpus từ phép đo nhỏ.

In [ ]:
phases=pd.DataFrame(run['phases']).T; display(phases)
phases['elapsed_sec'].plot.bar(title='Pha parser trong lượt Windows CPU',ylabel='Giây')
plt.tight_layout(); plt.show()
print('Runtime / publication xử lý mới:',run['runtime_sec']/run['processed_papers'])
print('Trung bình đã lưu:',run['avg_sec_per_publication'])

## 8.3 RAM, peak disk và output chuẩn

RSS chính/con được lấy mẫu như crawler. Peak disk theo dõi output workspace, có thể gồm
working JSON, checkpoint, samples và index tạm. `final_output_bytes` chỉ cộng các Parquet
của ba bảng chuẩn. Vì phạm vi khác nhau, final disk workspace lớn hơn kích thước ba
bảng chuẩn là điều có thể xảy ra; không được đồng nhất hai metric này.

In [ ]:
display(pd.Series(run['memory'],name='RAM').to_frame())
display(pd.Series(run['disk'],name='disk').to_frame())
display(pd.DataFrame([
    {'scope':'Peak output workspace được lấy mẫu','bytes':run['disk']['peak_bytes']},
    {'scope':'Final output workspace','bytes':run['disk']['final_bytes']},
    {'scope':'Ba Parquet store chuẩn','bytes':run['final_output_bytes']},
]).assign(MiB=lambda frame:frame['bytes']/1024**2))

### Diễn biến RSS và dung lượng đĩa

Các mẫu tài nguyên gốc của lượt Windows được tổng hợp theo cửa sổ 5 giây. Mỗi cửa sổ
giữ trung bình các mẫu RSS, RSS lớn nhất và disk lớn nhất đã quan sát. Trung bình RSS
trong cửa sổ là trung bình mẫu; chỉ số RAM trung bình toàn run ở trên dùng trọng số
thời gian và được giữ nguyên từ profiler. Điểm disk chỉ hiện ở cửa sổ có mẫu, không
điền thêm quan sát cho khoảng trống. Đơn vị MiB bằng 1.048.576 byte.

In [ ]:
profile = run.get('resource_profile')
if profile and profile['rows']:
    frame = pd.DataFrame(profile['rows'])
    fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
    rss = frame.dropna(subset=['rss_peak_mib'])
    axes[0].plot(rss['window_end_sec'], rss['rss_peak_mib'], label='Peak trong cửa sổ', color='#2f628e')
    axes[0].plot(rss['window_end_sec'], rss['rss_sample_mean_mib'], label='Mean mẫu trong cửa sổ', color='#7c9bb2')
    axes[0].set(xlabel='Giây từ đầu profiler', ylabel='RSS (MiB)', title='RAM của lượt parser Windows')
    axes[0].legend(fontsize=8)
    disk = frame.dropna(subset=['disk_peak_bytes'])
    axes[1].scatter(disk['window_end_sec'], disk['disk_peak_bytes']/1024**2, color='#b57b36', s=16)
    axes[1].set(xlabel='Giây từ đầu profiler', ylabel='Disk workspace (MiB)', title='Peak disk quan sát theo cửa sổ')
    plt.tight_layout(); plt.show()
    display(Markdown('**Hình M2.2. Tài nguyên parser theo thời gian.** '
                     f"Tổng hợp {profile['rss_samples']} mẫu RSS và {profile['disk_samples']} mẫu disk, "
                     f"cửa sổ {profile['window_sec']:g} giây. Đây là số đo Windows của snapshot, không phải Colab."))
else:
    print('Chưa có chuỗi mẫu tài nguyên cho run này; chỉ trình bày chỉ số tổng hợp đã lưu.')

## 8.4 Đo tiếp trên Colab và cập nhật đúng link

`colab_parser_submission.ipynb` trong source repository dùng đúng input snapshot và gói
runtime đã ghim SHA-256. Chọn CPU, tải hai ZIP đi kèm lên, chạy parser và validation, rồi
lấy archive output gồm stats, log, manifest và `time -v`. Không cần truy vấn arXiv/S2 khi
chỉ đo parser trên nguồn đã có.

Chỉ cập nhật số đo Book sau khi có output thật của Colab, kiểm tra scope/input hash và
kết quả validation. Nội dung được deploy vào repository Book hiện tại; URL nộp giữ nguyên.

[Notebook Colab cho parser](https://colab.research.google.com/github/NguyenQuocDat06-AI/arxiv-harvester/blob/develop/colab_parser_submission.ipynb)

## 8.5 Kết luận thực nghiệm

Pipeline đã tạo được ba store và validation cấu trúc trên snapshot, nhưng còn partial/error,
thiếu references và thiếu benchmark M2 Colab cuối. Việc làm đầy đủ phần trình bày không
thay thế các bước xử lý và đo còn thiếu. Các kết quả hiện tại được giữ làm mốc tiến độ
có thể đối chiếu, cập nhật sau trên cùng Book.

## Tài liệu và mã tham chiếu

- [Đặc tả M2](https://hcmus-cs-ta.github.io/lab-book/intro-ds/statement/parsing/).
- [arXiv API: cấu trúc truy vấn và Atom](https://info.arxiv.org/help/api/user-manual.html).
- [Quy định truy cập arXiv](https://info.arxiv.org/help/api/tou.html).
- [Semantic Scholar Academic Graph API](https://api.semanticscholar.org/api-docs/graph).
- Mã triển khai: [src/parser/main.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/parser/main.py).
- Mã triển khai: [src/crawler/profiler.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/src/crawler/profiler.py).
- Mã triển khai: [colab_parser_submission.ipynb](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/colab_parser_submission.ipynb).
- Mã triển khai: [tools/prepare_parser_colab.py](https://github.com/NguyenQuocDat06-AI/arxiv-harvester/blob/e8d409200fa3483bc52eb5a914c602c552def133/tools/prepare_parser_colab.py).